In [ ]:
import json
import pandas as pd
from datetime import datetime
from qkd_runner import run_multiple_trials

In [ ]:
conditions = [
    {'name': 'day1_72km', 'file': 'loss_data/day1_72km'},
    {'name': 'night1_72km', 'file': 'loss_data/night1_72km'},
]

# Simulation parameters
NUM_PAIRS = 500000
NUM_TRIALS = 5

In [ ]:
# NOTE this code fails; numpy can't handle N/A
# N/A caused by night time QBER = 0; so CASCADE efficiency not defined!!
# XXX json files cleaned seperately. this block's results are not reproducible!!

# Store all results
all_condition_results = []

# Run timestamp
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Starting simulation run: {run_timestamp}")
print(f"Parameters: {NUM_PAIRS} pairs, {NUM_TRIALS} trials per condition\n")
print("="*70)

for i, condition in enumerate(conditions, 1):
    print(f"\n[{i}/{len(conditions)}] Running {condition['name']}...")
    
    result = run_multiple_trials(
        condition_name=condition['name'],
        loss_file=condition['file'],
        num_pairs=NUM_PAIRS,
        num_trials=NUM_TRIALS
    )
    
    all_condition_results.append(result)
    
    # Print summary
    if result['num_secure'] > 0:
        print(f"  Summary: {result['num_secure']}/{NUM_TRIALS} secure runs")
        print(f"  Avg Final Key: {result['avg_final_key_bits']:.0f} ± {result['std_final_key_bits']:.0f} bits")
        print(f"  Avg Sifted Rate: {result['avg_sifted_rate_kbps']:.3f} ± {result['std_sifted_rate_kbps']:.3f} kbps")
        print(f"  Avg Reconciled Rate: {result['avg_reconciled_rate_kbps']:.3f} ± {result['std_reconciled_rate_kbps']:.3f} kbps")
        print(f"  Avg Final Rate: {result['avg_final_rate_kbps']:.3f} ± {result['std_final_rate_kbps']:.3f} kbps")
        print(f"  Avg CASCADE Efficiency: {result['avg_cascade_efficiency']:.3f}")
    else:
        print(f"  Summary: 0/{NUM_TRIALS} secure runs - condition too lossy")
    
    # Save intermediate results
    with open(f'results_{run_timestamp}_intermediate.json', 'w') as f:
        json.dump(all_condition_results, f, indent=2)

print("\n" + "="*70)
print("All simulations complete!")


In [ ]:
# NOTE code used to clean json 

import json

# Clean day1_72.json
with open('day1_72.json', 'r') as f:
    data = json.load(f)

# Extract loss_array once (it's the same everywhere)
loss_array = data[0]['all_trials'][0]['config']['loss_array']

# Remove from all trials
for item in data:
    item['loss_array'] = loss_array  # Store once at top level
    for trial in item['all_trials']:
        del trial['config']['loss_array']

with open('day1_72_cleaned.json', 'w') as f:
    json.dump(data, f, indent=2)

# Clean night1_72.json
with open('night1_72.json', 'r') as f:
    data = json.load(f)

loss_array = data[0]['all_trials'][0]['config']['loss_array']

for item in data:
    item['loss_array'] = loss_array
    for trial in item['all_trials']:
        del trial['config']['loss_array']

with open('night1_72_cleaned.json', 'w') as f:
    json.dump(data, f, indent=2)

print("Done! Files cleaned.")

In [ ]:
# Load the night1_72km results from the JSON file
import json

# Load the results from your earlier successful run
with open('night1_72.json', 'r') as f:
    data = json.load(f)

# Get night1_72km results (should be first in the list)
day_result = data[0]

# Format and print
print("="*70)
print(f"FINAL SUMMARY - {day_result['condition']}")
print("="*70)
print(f"Condition: {day_result['condition']}")
print(f"Secure Runs: {day_result['num_secure']}/{day_result['num_trials']}")
print(f"Emitted Pairs: {day_result['num_pairs']}")
print(f"Avg Alice Detections: {int(day_result['avg_alice_detections'])}")
print(f"Avg Bob Detections: {int(day_result['avg_bob_detections'])}")
print(f"Avg Coincidences: {int(day_result['avg_coincidences'])}")
print(f"Avg Sifted Key: {int(day_result['avg_sifted_key_bits'])} bits")
print(f"Avg QBER: {day_result['avg_qber']:.4f}")
print(f"Avg Reconciled Key: {int(day_result['avg_reconciled_key_bits'])} bits")
print(f"Avg Final Key: {int(day_result['avg_final_key_bits'])} ± {int(day_result['std_final_key_bits'])} bits")
print(f"Sifted Rate: {day_result['avg_sifted_rate_kbps']:.3f} ± {day_result['std_sifted_rate_kbps']:.3f} kbps")
print(f"Reconciled Rate: {day_result['avg_reconciled_rate_kbps']:.3f} ± {day_result['std_reconciled_rate_kbps']:.3f} kbps")
print(f"Final Rate: {day_result['avg_final_rate_kbps']:.3f} ± {day_result['std_final_rate_kbps']:.3f} kbps")
print(f"CASCADE Efficiency: N/A")
print("="*70)

FINAL SUMMARY - night1_72km
Condition: night1_72km
Secure Runs: 5/5
Emitted Pairs: 500000
Avg Alice Detections: 14080
Avg Bob Detections: 14124
Avg Coincidences: 400
Avg Sifted Key: 206 bits
Avg QBER: 0.0000
Avg Reconciled Key: 165 bits
Avg Final Key: 123 ± 4 bits
Sifted Rate: 0.041 ± 0.001 kbps
Reconciled Rate: 0.033 ± 0.001 kbps
Final Rate: 0.025 ± 0.001 kbps
CASCADE Efficiency: N/A


In [2]:
# Load the day1_72km results from the JSON file
import json

# Load the results from your earlier successful run
with open('day1_72.json', 'r') as f:
    data = json.load(f)

# Get day1_72km results (should be first in the list)
day_result = data[0]

# Format and print
print("="*70)
print(f"FINAL SUMMARY - {day_result['condition']}")
print("="*70)
print(f"Condition: {day_result['condition']}")
print(f"Secure Runs: {day_result['num_secure']}/{day_result['num_trials']}")
print(f"Emitted Pairs: {day_result['num_pairs']}")
print(f"Avg Alice Detections: {int(day_result['avg_alice_detections'])}")
print(f"Avg Bob Detections: {int(day_result['avg_bob_detections'])}")
print(f"Avg Coincidences: {int(day_result['avg_coincidences'])}")
print(f"Avg Sifted Key: {int(day_result['avg_sifted_key_bits'])} bits")
print(f"Avg QBER: {day_result['avg_qber']:.4f}")
print(f"Avg Reconciled Key: {int(day_result['avg_reconciled_key_bits'])} bits")
print(f"Avg Final Key: {int(day_result['avg_final_key_bits'])} ± {int(day_result['std_final_key_bits'])} bits")
print(f"Sifted Rate: {day_result['avg_sifted_rate_kbps']:.3f} ± {day_result['std_sifted_rate_kbps']:.3f} kbps")
print(f"Reconciled Rate: {day_result['avg_reconciled_rate_kbps']:.3f} ± {day_result['std_reconciled_rate_kbps']:.3f} kbps")
print(f"Final Rate: {day_result['avg_final_rate_kbps']:.3f} ± {day_result['std_final_rate_kbps']:.3f} kbps")
print(f"CASCADE Efficiency: {day_result['avg_cascade_efficiency']:.3f}")
print("="*70)

FINAL SUMMARY - day1_72km
Condition: day1_72km
Secure Runs: 5/5
Emitted Pairs: 500000
Avg Alice Detections: 39655
Avg Bob Detections: 39763
Avg Coincidences: 3143
Avg Sifted Key: 1580 bits
Avg QBER: 0.0792
Avg Reconciled Key: 1264 bits
Avg Final Key: 626 ± 34 bits
Sifted Rate: 0.316 ± 0.010 kbps
Reconciled Rate: 0.251 ± 0.008 kbps
Final Rate: 0.124 ± 0.007 kbps
CASCADE Efficiency: 1.213
